In [ ]:
hdfs

### Ringkasan Perintah HDFS yang Telah Dipelajari

| Perintah | Fungsi |
|---|---|
| `hdfs dfs -mkdir -p [path]` | Membuat direktori (dengan `-p`, otomatis membuat parent folder jika belum ada) |
| `hdfs dfs -ls [path]` | Menampilkan isi direktori |
| `hdfs dfs -put [lokal] [hdfs]` | Mengunggah berkas dari disk lokal ke HDFS |
| `hdfs dfs -get [hdfs] [lokal]` | Mengunduh berkas dari HDFS ke disk lokal |
| `hdfs dfs -cat [path]` | Menampilkan isi berkas teks di HDFS |
| `hdfs dfs -du -h [path]` | Menampilkan ukuran penggunaan disk (format *human-readable*) |
| `hdfs dfs -count [path]` | Menghitung jumlah direktori/berkas dan total ukuran |
| `hdfs dfs -rm [path]` | Menghapus berkas (tambahkan `-r` untuk menghapus direktori beserta isinya) |


### Instruksi Pengerjaan

Buat **notebook BARU** bernama **`Tugas3_[NIM]_[Nama Lengkap].ipynb`**, letakkan di folder `~/praktikum-bigdata` yang sama dengan ketiga berkas CSV di atas, lalu kerjakan seluruh bagian **A sampai E** berikut secara berurutan.

---

**A. Membangun Struktur Direktori HDFS** *(bobot 15%)*

Buatlah struktur direktori berikut di HDFS:
```
/user/[username]/ecommerce/raw
/user/[username]/ecommerce/processed
```
Tampilkan hasilnya untuk membuktikan struktur sudah terbentuk.

**B. Mengunggah Data Mentah ke HDFS** *(bobot 15%)*

Unggah **ketiga berkas CSV** (`transaksi_magelang.csv`, `transaksi_yogyakarta.csv`, `transaksi_semarang.csv`) ke direktori `/user/[username]/ecommerce/raw` di HDFS. Tampilkan isi direktori tersebut sebagai bukti ketiga berkas berhasil terunggah, lengkap dengan ukurannya.

**C. Membaca Kembali dan Menggabungkan Data dari HDFS** *(bobot 25%)*

Menggunakan Python, baca kembali **ketiga berkas dari HDFS** (bukan dari disk lokal!). Gabungkan ketiganya menjadi **satu DataFrame**, lalu buktikan hasil gabungan berisi transaksi dari ketiga kota (tampilkan `value_counts()` pada kolom `kota`).

**D. Mengolah dan Download Hasil ke Direktori `processed`** *(bobot 25%)*

Pada DataFrame gabungan hasil bagian C:
1. Tambahkan kolom `total_pendapatan` (`unit_terjual x harga_satuan`).
2. Buatlah **satu tabel ringkasan** (`groupby`) yang menampilkan total pendapatan per kota **dan** per kategori sekaligus.
3. Simpan **dua** hasil berikut sebagai berkas CSV baru di disk lokal, lalu unggah keduanya ke `/user/[username]/ecommerce/processed` di HDFS:
   - `data_gabungan_bersih.csv` (seluruh data ketiga kota yang telah digabung + kolom `total_pendapatan`)
   - `ringkasan_kota_kategori.csv` (tabel ringkasan dari langkah 2)

**E. Dokumentasi dan Refleksi** *(bobot 20%)*

Sertakan pada notebook anda:
1. **Screenshot** halaman NameNode Web UI (http://localhost:9870, menu *Utilities → Browse the file system*) yang menunjukkan struktur folder `/user/[username]/ecommerce/` beserta isinya (paste gambar ke dalam markdown cell, atau lampirkan terpisah jika format pengumpulan berupa arsip/zip.
2. Tulisan reflektif (**minimal 100 kata**) pada markdown cell yang menjawab: *Apa keuntungan menyimpan data mentah (raw) terpisah dari data olahan (processed) di HDFS, dibandingkan menyimpan semuanya bercampur dalam satu folder?*

---

In [4]:
!hdfs dfs -mkdir -p /user/haris/ecommerce/raw
!hdfs dfs -mkdir -p /user/haris/ecommerce/processed

In [6]:
!hdfs dfs -ls /user/haris/ecommerce/

Found 2 items
drwxr-xr-x   - haris supergroup          0 2026-09-12 19:36 /user/haris/ecommerce/processed
drwxr-xr-x   - haris supergroup          0 2026-09-12 19:36 /user/haris/ecommerce/raw


In [7]:
# Mengunggah ketiga berkas CSV dari disk lokal VM ke HDFS raw
!hdfs dfs -put transaksi_magelang.csv /user/haris/ecommerce/raw/
!hdfs dfs -put transaksi_yogyakarta.csv /user/haris/ecommerce/raw/
!hdfs dfs -put transaksi_semarang.csv /user/haris/ecommerce/raw/

# Verifikasi isi direktori raw di HDFS beserta ukuran berkasnya (-h untuk human-readable)
!hdfs dfs -ls -h /user/haris/ecommerce/raw

Found 3 items
-rw-r--r--   1 haris supergroup     12.0 K 2026-09-12 19:44 /user/haris/ecommerce/raw/transaksi_magelang.csv
-rw-r--r--   1 haris supergroup     11.9 K 2026-09-12 19:44 /user/haris/ecommerce/raw/transaksi_semarang.csv
-rw-r--r--   1 haris supergroup     12.4 K 2026-09-12 19:44 /user/haris/ecommerce/raw/transaksi_yogyakarta.csv


In [8]:
import io
import pandas as pd

# Daftar berkas cabang di HDFS
berkas_cabang = [
    "transaksi_magelang.csv",
    "transaksi_yogyakarta.csv",
    "transaksi_semarang.csv",
]
list_df = []

for file in berkas_cabang:
    hdfs_path = f"/user/haris/ecommerce/raw/{file}"

    # Membaca isi berkas langsung dari HDFS tanpa mengunduhnya ke disk lokal
    content_hdfs = !hdfs dfs -cat {hdfs_path}

    # Mengubah output text HDFS menjadi pandas DataFrame
    df_temp = pd.read_csv(io.StringIO("\n".join(content_hdfs)))
    list_df.append(df_temp)

# Menggabungkan ketiga DataFrame cabang menjadi satu DataFrame utama
df_gabungan = pd.concat(list_df, ignore_index=True)

# Membuktikan hasil gabungan berisi transaksi dari ketiga kota
print("=== Jumlah Transaksi per Kota ===")
print(df_gabungan["kota"].value_counts())
print("\n=== Dimensi Data Gabungan ===")
print(f"Total Baris & Kolom: {df_gabungan.shape}")
df_gabungan.head()

=== Jumlah Transaksi per Kota ===
kota
Magelang      200
Yogyakarta    200
Semarang      200
Name: count, dtype: int64

=== Dimensi Data Gabungan ===
Total Baris & Kolom: (600, 7)


,order_id,tanggal,kategori,unit_terjual,harga_satuan,metode_pembayaran,kota
0,MAG-2000,2026-08-12,Fashion,1,50000,Transfer Bank,Magelang
1,MAG-2001,2026-08-18,Elektronik,7,25000,COD,Magelang
2,MAG-2002,2026-08-07,Elektronik,7,25000,E-Wallet,Magelang
3,MAG-2003,2026-08-24,Rumah Tangga,6,25000,COD,Magelang
4,MAG-2004,2026-08-30,Fashion,7,100000,Transfer Bank,Magelang


In [9]:
# 1. Menambahkan kolom total_pendapatan
df_gabungan["total_pendapatan"] = (
    df_gabungan["unit_terjual"] * df_gabungan["harga_satuan"]
)

# 2. Membuat tabel ringkasan total pendapatan per kota dan per kategori
df_ringkasan = (
    df_gabungan.groupby(["kota", "kategori"])["total_pendapatan"]
    .sum()
    .reset_index()
)

# Tampilkan ringkasan data
print("=== Ringkasan Total Pendapatan per Kota & Kategori ===")
display(df_ringkasan)

# 3. Menyimpan kedua hasil ke berkas CSV di disk lokal
df_gabungan.to_csv("data_gabungan_bersih.csv", index=False)
df_ringkasan.to_csv("ringkasan_kota_kategori.csv", index=False)
print("\nBerkas CSV lokal berhasil disimpan.")

=== Ringkasan Total Pendapatan per Kota & Kategori ===


,kota,kategori,total_pendapatan
0,Magelang,Elektronik,18775000
1,Magelang,Fashion,27750000
2,Magelang,Kesehatan & Kecantikan,17375000
3,Magelang,Makanan & Minuman,17525000
4,Magelang,Rumah Tangga,12200000
5,Semarang,Elektronik,21425000
6,Semarang,Fashion,26425000
7,Semarang,Kesehatan & Kecantikan,13525000
8,Semarang,Makanan & Minuman,19750000
9,Semarang,Rumah Tangga,10975000



Berkas CSV lokal berhasil disimpan.


In [10]:
# Mengunggah kedua berkas hasil olahan ke direktori processed di HDFS
!hdfs dfs -put data_gabungan_bersih.csv /user/haris/ecommerce/processed/
!hdfs dfs -put ringkasan_kota_kategori.csv /user/haris/ecommerce/processed/

# Verifikasi isi direktori processed di HDFS
!hdfs dfs -ls -h /user/haris/ecommerce/processed

Found 2 items
-rw-r--r--   1 haris supergroup     40.3 K 2026-09-12 19:48 /user/haris/ecommerce/processed/data_gabungan_bersih.csv
-rw-r--r--   1 haris supergroup        530 2026-09-12 19:48 /user/haris/ecommerce/processed/ringkasan_kota_kategori.csv
